# Semana 2: El Universo del Dato: Infraestructura, Gobernanza y Calidad
## Notebook Conceptual (NB1) – Datos, Hardware y Gobernanza con Python

### Propósito de la sesión
Comprender el ciclo de vida del dato desde su generación hasta su explotación, incluyendo el soporte físico (hardware), los roles involucrados en su gestión, y los marcos de gobernanza que garantizan su calidad, privacidad y trazabilidad. Estableceremos la conexión entre estos conceptos y las herramientas prácticas que se usarán en el curso.

### Objetivos de aprendizaje
*   Distinguir entre **dato, información y metadata**.
*   Simular el comportamiento de **hardware** (RAM, disco) y su impacto en búsquedas.
*   Conocer los **roles** en la gestión de datos: DBA, Data Engineer, Data Scientist.
*   Aplicar **Great Expectations** para validar calidad de datos.
*   Construir y visualizar un grafo de **linaje de datos (data lineage)** con networkx.

## Configuración Inicial

Importamos las librerías necesarias y configuramos el entorno.

In [ ]:
# Importamos librerías
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
import timeit
import time
import random
import hashlib
from datetime import datetime

# Intentamos instalar Great Expectations (si no está)
try:
    import great_expectations as ge
    print("✅ Great Expectations ya está instalado.")
except ImportError:
    print("Instalando Great Expectations...")
    !pip install great_expectations
    import great_expectations as ge
    print("✅ Great Expectations instalado.")

# Configuración de matplotlib
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (12, 6)

# Fijamos semilla para reproducibilidad
np.random.seed(42)
random.seed(42)

print("\n🎯 Librerías importadas correctamente.")

---
## Actividad 1: Dato, Información y Metadata

### 1.1. Creación de un DataFrame con datos crudos

Simulamos transacciones de una tienda online.

In [ ]:
# Generamos datos de ejemplo
n = 1000
data = {
    'transaction_id': range(1000, 1000+n),
    'cliente_id': np.random.randint(1, 101, n),
    'monto': np.random.uniform(10, 500, n).round(2),
    'fecha': pd.date_range('2025-01-01', periods=n, freq='H'),
    'producto': np.random.choice(['Laptop', 'Mouse', 'Teclado', 'Monitor', 'Auriculares'], n),
    'email': [f"cliente{random.randint(1,100)}@email.com" for _ in range(n)]
}

df = pd.DataFrame(data)
print("Primeras 5 transacciones:")
display(df.head())

### 1.2. Metadata del DataFrame

La metadata son los datos que describen los datos. En pandas, la obtenemos con `.info()`, `.describe()`, `.dtypes`.

In [ ]:
print("🔷 Metadata del DataFrame:")
print("\nInformación general (.info()):")
df.info()

print("\nTipos de datos (.dtypes):")
print(df.dtypes)

print("\nEstadísticas descriptivas (.describe()):")
display(df.describe(include='all'))

### 1.3. Dato vs Información

*   **Dato**: 1010, 150.50, 'Laptop' (valores crudos).
*   **Información**: "El cliente 1010 compró una Laptop por $150.50 el 2025-01-01".
*   **Metadata**: "La columna 'monto' es de tipo float64 y representa el valor de la transacción en dólares".

### 1.4. Generación de logs simulados

Los logs son esenciales para auditoría y trazabilidad. Simularemos un log de accesos.

In [ ]:
def generate_log_entry():
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    user = f"user_{random.randint(1,50)}"
    action = random.choice(['SELECT', 'INSERT', 'UPDATE', 'DELETE'])
    table = random.choice(['clientes', 'productos', 'ventas'])
    status = random.choice(['SUCCESS', 'ERROR'])
    return f"{timestamp} | {user} | {action} | {table} | {status}"

logs = [generate_log_entry() for _ in range(20)]
print("\n🔶 Logs simulados:")
for log in logs:
    print(log)

---
## Actividad 2: Simulación de Hardware y Búsqueda

Recordamos los índices vistos en la Semana 1: B-Tree (búsqueda O(log n)) y Hash (búsqueda O(1)).

### 2.1. Comparación de tiempos: lista (simula disco) vs diccionario (simula RAM)

In [ ]:
# Creamos una lista grande (simula datos en disco)
size = 100000
data_list = list(range(size))

# Creamos un diccionario (simula índice en RAM)
data_dict = {i: i for i in range(size)}

# Función para buscar en lista (O(n))
def search_list():
    return 99999 in data_list

# Función para buscar en diccionario (O(1))
def search_dict():
    return 99999 in data_dict

# Medimos tiempos
t_list = timeit.timeit(search_list, number=1000)
t_dict = timeit.timeit(search_dict, number=1000)

print(f"🔷 Tiempo de 1000 búsquedas en lista (simula disco): {t_list:.6f} s")
print(f"🔶 Tiempo de 1000 búsquedas en diccionario (simula RAM/index): {t_dict:.6f} s")
print(f"📊 El diccionario es {t_list/t_dict:.1f} veces más rápido.")

### 2.2. Simulación de un Buffer Pool (caché LRU simple)

El buffer pool mantiene en RAM las páginas más usadas. Simulamos un caché con un diccionario y un límite de tamaño.

In [ ]:
class SimpleLRUCache:
    def __init__(self, capacity=5):
        self.capacity = capacity
        self.cache = {}
        self.access_count = {}  # simulamos orden de acceso
        self.counter = 0
    
    def get(self, key):
        if key in self.cache:
            self.access_count[key] = self.counter
            self.counter += 1
            return self.cache[key]
        return None
    
    def put(self, key, value):
        if len(self.cache) >= self.capacity and key not in self.cache:
            # Encontrar la clave menos recientemente usada
            lru_key = min(self.access_count, key=self.access_count.get)
            del self.cache[lru_key]
            del self.access_count[lru_key]
        self.cache[key] = value
        self.access_count[key] = self.counter
        self.counter += 1
    
    def __str__(self):
        return str(self.cache)

# Simulamos accesos
cache = SimpleLRUCache(capacity=3)
cache.put('página1', 'datos1')
cache.put('página2', 'datos2')
cache.put('página3', 'datos3')
print("Cache inicial:", cache)

# Accedemos a página1 (se marca como reciente)
cache.get('página1')
print("Acceso a página1:", cache)

# Insertamos página4 (debe expulsar la menos reciente: página2)
cache.put('página4', 'datos4')
print("Insertar página4 (expulsa la menos usada):", cache)

---
## Actividad 3: Roles en la Gestión de Datos

Simulamos tareas de diferentes roles:
*   **DBA**: Crea índices en SQLite.
*   **Data Engineer**: Construye un pipeline ETL simple.
*   **Data Scientist / Analista**: Genera un dashboard básico.

In [ ]:
# Conectamos a SQLite en memoria
import sqlite3

conn = sqlite3.connect(':memory:')
cursor = conn.cursor()

# DBA: crea tabla e índices
cursor.execute('''
    CREATE TABLE ventas (
        id INTEGER PRIMARY KEY,
        cliente_id INTEGER,
        monto REAL,
        fecha TEXT
    )
''')

# Insertamos datos desde el DataFrame
df[['cliente_id', 'monto', 'fecha']].to_sql('ventas', conn, if_exists='append', index=False)

# DBA crea índice en cliente_id
cursor.execute('CREATE INDEX idx_cliente ON ventas(cliente_id)')
print("✅ DBA: Índice creado en cliente_id.")

# Data Engineer: pipeline ETL (extrae, transforma, carga)
# Extrae: consulta datos
df_ventas = pd.read_sql_query("SELECT * FROM ventas", conn)
# Transforma: calcula total por cliente
total_por_cliente = df_ventas.groupby('cliente_id')['monto'].sum().reset_index()
# Carga (simulada, guardamos en CSV)
total_por_cliente.to_csv('total_por_cliente.csv', index=False)
print("✅ Data Engineer: Pipeline ETL completado. Total por cliente calculado.")

# Data Scientist: dashboard simple
plt.figure(figsize=(10, 5))
plt.bar(total_por_cliente['cliente_id'].head(10), total_por_cliente['monto'].head(10))
plt.xlabel('Cliente ID')
plt.ylabel('Total gastado')
plt.title('Top 10 clientes por gasto total')
plt.show()
print("✅ Data Scientist: Dashboard generado.")

---
## Actividad 4: Calidad de Datos con Great Expectations

Great Expectations es una herramienta para validar, documentar y probar la calidad de los datos.

### 4.1. Crear un conjunto de datos con errores comunes

In [ ]:
# Datos con errores
data_con_errores = {
    'id': [1, 2, 3, 4, 5, 6],
    'edad': [25, -5, 30, None, 150, 28],
    'email': ['juan@mail.com', 'ana@mail', 'pedro@mail.com', 'laura@mail.com', None, 'carlos@mail.com'],
    'salario': [3000, 4500, None, 2800, 5000, 6000]
}
df_errors = pd.DataFrame(data_con_errores)

# Convertir a Great Expectations DataFrame
ge_df = ge.from_pandas(df_errors)

print("📊 Datos con errores:")
display(df_errors)

### 4.2. Definir expectativas y validar

In [ ]:
# Definir expectativas
expectations = [
    ge_df.expect_column_values_to_be_between('edad', min_value=0, max_value=120),
    ge_df.expect_column_values_to_not_be_null('id'),
    ge_df.expect_column_values_to_match_regex('email', r'^[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+$'),
    ge_df.expect_column_values_to_not_be_null('salario'),
    ge_df.expect_column_values_to_be_between('salario', min_value=1000, max_value=10000)
]

# Mostrar resultados
print("🔍 Resultados de validación:")
for exp in expectations:
    success = exp['success']
    result = exp['result']
    print(f"  {exp['expectation_type']}: {'✅' if success else '❌'} - {result.get('unexpected_count', 0)} valores inesperados")

# Generar reporte HTML (opcional, se guarda en el entorno)
# ge_df.save_expectation_suite('mi_suite')
# ge_df.validate().to_json("resultados.json")
print("\n📌 Para generar un reporte HTML interactivo, puedes usar 'ge_df.validate().to_json()' y herramientas de Great Expectations.")

---
## Actividad 5: Linaje de Datos (Data Lineage) con NetworkX

El linaje de datos permite rastrear el origen y las transformaciones de los datos. Construimos un grafo dirigido.

In [ ]:
# Crear grafo dirigido
G = nx.DiGraph()

# Añadir nodos (fuentes, procesos, destinos)
sources = ['API Clientes', 'DB Transacciones', 'Archivo CSV']
processes = ['Limpieza', 'Agregación', 'Feature Engineering']
destinations = ['Dashboard', 'Modelo ML']

for node in sources + processes + destinations:
    G.add_node(node)

# Añadir aristas (flujo de datos)
edges = [
    ('API Clientes', 'Limpieza'),
    ('DB Transacciones', 'Limpieza'),
    ('Archivo CSV', 'Limpieza'),
    ('Limpieza', 'Agregación'),
    ('Agregación', 'Feature Engineering'),
    ('Feature Engineering', 'Dashboard'),
    ('Feature Engineering', 'Modelo ML')
]
G.add_edges_from(edges)

# Visualizar
plt.figure(figsize=(12, 8))
pos = nx.spring_layout(G, seed=42, k=2)
nx.draw_networkx_nodes(G, pos, node_color='lightblue', node_size=2000)
nx.draw_networkx_edges(G, pos, edge_color='gray', arrows=True, arrowsize=20)
nx.draw_networkx_labels(G, pos, font_size=12)
plt.title('Grafo de Linaje de Datos', fontsize=16)
plt.axis('off')
plt.show()

print("📌 El linaje permite auditar: ¿De dónde vino este dato? ¿Qué transformaciones sufrió?")
print("📌 Es crucial para depuración, cumplimiento normativo y entender el impacto de cambios.")

---
## Ejercicios para el Estudiante

1.  **Metadata adicional**: Añade una columna de `fecha_nacimiento` al DataFrame y genera metadata que describa su formato y rango.

2.  **Simulación de buffer pool**: Modifica la clase LRU para que muestre el número de hits y misses. Simula una secuencia de accesos y calcula la tasa de aciertos.

3.  **Great Expectations**: Añade más expectativas al conjunto de datos (por ejemplo, que los emails sean únicos). Genera un reporte HTML exploratorio.

4.  **Linaje extendido**: Añade nodos y aristas adicionales al grafo de linaje, por ejemplo, un proceso de validación de calidad antes del modelo.

5.  **Reflexión**: ¿Por qué es importante la calidad de datos para un modelo de IA? ¿Qué pasaría si entrenamos un modelo con datos que no pasan las expectativas definidas?

---
## Conclusiones de la Sesión

*   Hemos distinguido entre **dato, información y metadata**, y visto cómo acceder a la metadata en pandas.
*   Simulamos el impacto del hardware en búsquedas: listas (disco) vs diccionarios (RAM/índices).
*   Introdujimos el concepto de **buffer pool** con una simulación LRU.
*   Exploramos los **roles** típicos en gestión de datos (DBA, Data Engineer, Data Scientist) mediante tareas simuladas.
*   Aplicamos **Great Expectations** para validar calidad de datos con expectativas sobre edad, email, etc.
*   Construimos un grafo de **linaje de datos** con networkx, visualizando el flujo desde fuentes hasta consumo.

¡Ahora comprendes el ciclo de vida del dato y la importancia de su gobernanza!